# Drone Detector - 01 Data Exploration
Deze notebook is Colab-ready en helpt je bij Fase 1 (classificatie) en Fase 2 (bbox detectie).

In [ ]:
# Google Colab setup
import os

# Uncomment bij gebruik in Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/drone_detector')

print('Huidige map:', os.getcwd())

In [ ]:
# Kaggle dataset download (Colab)
# !pip install kaggle
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <dataset-name> -p data/raw/ --unzip

In [ ]:
# Stap 1: Imports
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
# Stap 2: Laad configuratie
with open('configs/config.yaml', 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

IMAGE_SIZE = tuple(config['image_size'])
BATCH_SIZE = config['batch_size']
EPOCHS = config['epochs']
LR = config['learning_rate']

DATA_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
MODEL_SAVE_DIR = Path('models')
REPORTS_DIR = Path('reports/figures')

print('Config geladen:', config)

## Fase 1 - Classificatie Data Check

In [ ]:
# Stap 3: Overzicht van train/val/test mappen
splits = ['train', 'val', 'test']
summary_rows = []

for split in splits:
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        continue

    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            count = len(list(class_dir.glob('*')))
            summary_rows.append({
                'split': split,
                'class_name': class_dir.name,
                'n_images': count
            })

df_counts = pd.DataFrame(summary_rows)
display(df_counts.sort_values(['split', 'class_name']) if not df_counts.empty else 'Geen class mappen gevonden')

In [ ]:
# Stap 4: Visualiseer klasseverdeling
if 'df_counts' in globals() and not df_counts.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=df_counts, x='split', y='n_images', hue='class_name')
    plt.title('Klasseverdeling per split')
    plt.tight_layout()
    plt.show()
else:
    print('Geen classificatie-overzicht beschikbaar.')

## Fase 2 - Bounding Box Annotatie Check

In [ ]:
# Stap 5: Controleer annotatiebestanden voor bbox detectie
annotation_candidates = [
    DATA_DIR / 'train_annotations.csv',
    DATA_DIR / 'val_annotations.csv',
    DATA_DIR / 'test_annotations.csv',
]

required_cols = {'image_path', 'label', 'xmin', 'ymin', 'xmax', 'ymax'}

for csv_path in annotation_candidates:
    if not csv_path.exists():
        print(f'Niet gevonden: {csv_path}')
        continue

    df_ann = pd.read_csv(csv_path)
    missing = required_cols.difference(df_ann.columns)
    print(f'\nBestand: {csv_path}')
    print(f'Rijen: {len(df_ann)}')
    if missing:
        print('Ontbrekende kolommen:', sorted(missing))
    else:
        print('Alle vereiste kolommen aanwezig.')
        display(df_ann.head(3))

## Volgende Stap
Gebruik nu src/models/train_model.py met --phase classification of --phase detection om respectievelijk Fase 1 of Fase 2 te trainen.